# TripoSR 성능 테스트
단일 이미지 → 3D 모델 변환

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# TripoSR 설치
!git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip install -r requirements.txt

In [ ]:
# 테스트 이미지 업로드
from google.colab import files
uploaded = files.upload()  # 이미지 파일 선택

import os
image_path = list(uploaded.keys())[0]
print(f'업로드된 이미지: {image_path}')

In [ ]:
# 배경 제거 (선택사항 - 결과 품질 향상)
# cupy 충돌 제거 후 CPU 전용 rembg 설치
!pip uninstall -y cupy-cuda11x cupy-cuda12x cupy 2>/dev/null; echo "cupy 제거 완료"
!pip install onnxruntime -q
!pip install rembg -q

# rembg가 cupy를 찾지 않도록 환경변수 설정
import os
os.environ["REMBG_DISABLE_GPU"] = "1"

from rembg import remove
from PIL import Image

input_img = Image.open(image_path)
output_img = remove(input_img)
bg_removed_path = 'input_no_bg.png'
output_img.save(bg_removed_path)

# 원본 vs 배경제거 비교
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(input_img)
axes[0].set_title('원본')
axes[0].axis('off')
axes[1].imshow(output_img)
axes[1].set_title('배경 제거')
axes[1].axis('off')
plt.show()

In [ ]:
# TripoSR 실행
import os
os.makedirs('output/0', exist_ok=True)

!python run.py {bg_removed_path} --output-dir output --no-remove-bg

print('완료! output/ 폴더에 결과 저장됨')

In [ ]:
# 결과 파일 확인
import glob
output_files = glob.glob('output/**/*', recursive=True)
for f in output_files:
    print(f)

In [ ]:
# 3D 결과물 시각화 (mesh 미리보기)
!pip install trimesh -q

import trimesh
import glob

obj_files = glob.glob('output/**/*.obj', recursive=True)
if obj_files:
    mesh = trimesh.load(obj_files[0])
    print(f'버텍스 수: {len(mesh.vertices)}')
    print(f'면(face) 수: {len(mesh.faces)}')
    mesh.show()  # 3D 뷰어
else:
    print('obj 파일 없음, 다른 포맷 확인')
    glb_files = glob.glob('output/**/*.glb', recursive=True)
    print('glb 파일:', glb_files)

In [ ]:
# 결과 파일 다운로드
from google.colab import files
import glob

for f in glob.glob('output/**/*.*', recursive=True):
    files.download(f)